# Spotify Song Recommendation And Popularity Modeling

This notebook builds the modeling layer for the composer dashboard. It uses `spotify_cleaned_eda.csv` and creates three outputs:

1. Similar existing songs based on audio-feature similarity.
2. Fame-free base popularity prediction using composition features only.
3. Artist-fit recommendations using style similarity plus estimated artist lift.

The workflow now compares several popularity models before selecting the final base model.

## 1. Setup

Use the project virtual environment if possible: `.venv\Scripts\python.exe`.

In [4]:
import sys
from pathlib import Path

PROJECT_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "src" / "spotify_train_models.py").exists())
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

import json
import joblib
import numpy as np
import pandas as pd

from spotify_train_models import (
    BASE_POPULARITY_MODEL_NAME,
    DATA_PATH,
    MODEL_DIR,
    NUMERIC_FEATURES,
    POPULARITY_BANDS,
    POPULARITY_FEATURES,
    RECOMMENDER_FEATURES,
    build_artist_profiles,
    build_model_metadata,
    build_recommender,
    evaluate_popularity_models,
    load_modeling_data,
    predict_base_popularity,
    recommend_artists,
    recommend_songs,
    train_final_popularity_model,
)

## 2. Load Modeling Data

This starts from the cleaned EDA dataset, then applies final modeling checks: required columns, numeric coercion, valid tempo/duration, and non-missing metadata.

In [5]:
df = load_modeling_data(DATA_PATH)

print(f"Rows after modeling checks: {len(df):,}")
print(f"Unique tracks: {df['track_id'].nunique():,}")
print(f"Genres: {df['track_genre'].nunique():,}")
df.head()

Rows after modeling checks: 113,842
Unique tracks: 89,583
Genres: 114


,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,...,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre,duration_min,explicit_int
0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,1,...,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic,3.844433,0
1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,1,...,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic,2.493500,0
2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,0,...,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic,3.513767,0
3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,0,...,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic,3.365550,0
4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,2,...,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic,3.314217,0


## 3. Feature Design

The base popularity model is intentionally fame-free. It excludes `artists`, `album_name`, `track_name`, and `track_id`. Artist influence is handled later as a separate fit/lift layer.

In [6]:
print("Recommender features:")
print(RECOMMENDER_FEATURES)

print("Popularity model features:")
print(POPULARITY_FEATURES)

print("Popularity bands:")
print(pd.DataFrame(POPULARITY_BANDS))

Recommender features:
['danceability', 'energy', 'loudness', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_min', 'explicit_int']
Popularity model features:
['danceability', 'energy', 'loudness', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_min', 'explicit_int', 'key', 'mode', 'time_signature', 'track_genre']
Popularity bands:
      label  min  max
0       Low    0   34
1  Moderate   35   59
2      High   60  100


## 4. Similar Song Recommender

This model uses scaled audio features and cosine distance. If the user selects a genre, the recommendation helper searches inside that genre first.

In [7]:
recommender_scaler, recommender, song_catalog = build_recommender(df)

print(f"Song catalog rows after track_id deduplication: {len(song_catalog):,}")
song_catalog.head()

Song catalog rows after track_id deduplication: 89,583


,track_id,track_name,artists,album_name,track_genre,popularity,danceability,energy,valence,tempo,duration_min,loudness,speechiness,acousticness,instrumentalness,liveness,explicit_int
0,5SuOikwiRyPMVoIQDJUgSV,Comedy,Gen Hoshino,Comedy,acoustic,73,0.676,0.4610,0.715,87.917,3.844433,-6.746,0.1430,0.0322,0.000001,0.3580,0
1,4qPNDBW1i3p13qLCt0Ki3A,Ghost - Acoustic,Ben Woodward,Ghost (Acoustic),acoustic,55,0.420,0.1660,0.267,77.489,2.493500,-17.235,0.0763,0.9240,0.000006,0.1010,0
2,1iJBSr7s7jYXzM8EGcbK5b,To Begin Again,Ingrid Michaelson;ZAYN,To Begin Again,acoustic,57,0.438,0.3590,0.120,76.332,3.513767,-9.734,0.0557,0.2100,0.000000,0.1170,0
3,6lfxq3CG4xtTiEg7opyCyx,Can't Help Falling In Love,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,acoustic,71,0.266,0.0596,0.143,181.740,3.365550,-18.515,0.0363,0.9050,0.000071,0.1320,0
4,5vjLSffimiIP26QG5WcN2K,Hold On,Chord Overstreet,Hold On,acoustic,82,0.618,0.4430,0.167,119.949,3.314217,-9.681,0.0526,0.4690,0.000000,0.0829,0


## 5. Popularity Modeling Strategy

Popularity is a social and commercial outcome, so audio features alone should not be expected to explain all variance. We compare several models, then use the best practical baseline for the composer-facing fame-free score.

In [8]:
model_comparison = evaluate_popularity_models(df)
model_comparison[["model", "mae", "rmse", "r2", "train_rows", "test_rows"]].round(4)

,model,mae,rmse,r2,train_rows,test_rows
0,HistGB_tuned,12.9933,17.7932,0.3624,91072,22770
1,RandomForest,13.6943,18.0071,0.3470,91072,22770
2,ExtraTrees,13.4778,18.2965,0.3259,91072,22770
3,HistGB_current,13.7400,18.3742,0.3201,91072,22770
4,Ridge,14.1304,19.1426,0.2621,91072,22770
5,DummyMean,18.8959,22.2859,-0.0002,91072,22770


## 6. Select Final Base Popularity Model

The selected model is the tuned histogram gradient boosting regressor. It performed best in the comparison split while keeping the input features fame-free.

In [9]:
selected_model_name = BASE_POPULARITY_MODEL_NAME
selected_metrics = model_comparison[model_comparison["model"] == selected_model_name].iloc[0]

print(f"Selected model: {selected_model_name}")
print(selected_metrics[["mae", "rmse", "r2", "unique_train_tracks", "unique_test_tracks"]])

Selected model: HistGB_tuned
mae                    12.993259
rmse                   17.793215
r2                      0.362437
unique_train_tracks        71666
unique_test_tracks         17917
Name: 0, dtype: object


## 7. Train Final Base Popularity Model

After evaluation, retrain the selected model on the full modeling dataset so the dashboard uses all available training data.

In [10]:
final_popularity_model = train_final_popularity_model(df, selected_model_name)
print("Final fame-free base popularity model trained.")

Final fame-free base popularity model trained.


## 8. Artist Fit Profiles

Artist fit is estimated separately from base popularity. The model calculates historical artist residual lift: actual popularity minus the base composition-only prediction. This avoids contaminating the base composer score with artist fame.

In [11]:
artist_profiles = build_artist_profiles(df, final_popularity_model, min_tracks=5)

print(f"Artist profiles: {len(artist_profiles):,}")
artist_profiles.head(10)

Artist profiles: 6,040


,artist,track_count,avg_popularity,artist_lift,main_genre,genre_count,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_min,explicit_int,artist_lift_shrunk
0,Frank Ocean,21,73.956522,36.940260,soul,4,0.630826,0.467383,-8.559391,0.108665,0.514494,0.000576,0.217178,0.432783,115.553696,4.118750,0.652174,25.024047
1,The Neighbourhood,27,75.583333,34.110071,alt-rock,4,0.542050,0.645433,-6.650083,0.039015,0.077243,0.110313,0.226470,0.354930,108.388533,3.751536,0.083333,24.891133
2,The 1975,23,70.833333,33.150202,alt-rock,3,0.612738,0.741786,-6.311643,0.043771,0.247750,0.065705,0.173671,0.630498,120.571310,4.283391,0.309524,23.104686
3,Rammstein,76,63.945652,25.879471,industrial,4,0.568391,0.834870,-5.520402,0.054382,0.018033,0.136614,0.175618,0.394809,125.047174,4.181375,0.032609,22.870230
4,Enrique Iglesias,21,56.370370,33.449978,latin,6,0.661296,0.756111,-4.753333,0.061437,0.235090,0.000011,0.128870,0.604148,118.067593,3.738967,0.111111,22.659663
5,Kane Brown,9,70.111111,44.268047,country,2,0.597222,0.670556,-5.939000,0.043544,0.214780,0.000003,0.215556,0.517222,114.455333,2.989426,0.000000,20.969075
6,Halsey,37,71.466667,25.743769,indie-pop,13,0.641378,0.638567,-6.054500,0.068420,0.184364,0.008326,0.167986,0.466244,121.828144,3.379162,0.355556,20.266371
7,Ariana Grande,18,72.238095,31.039401,dance,6,0.655524,0.592095,-6.352238,0.113624,0.274256,0.000007,0.174038,0.457190,131.469667,3.407214,0.571429,19.953901
8,Farruko,20,69.822222,29.928518,latino,6,0.709444,0.758711,-4.654089,0.065753,0.061018,0.057675,0.183191,0.478000,137.005111,4.149806,0.666667,19.952346
9,Adele,51,67.573333,23.313894,british,2,0.544453,0.487133,-7.033880,0.045175,0.473450,0.002483,0.127855,0.329521,122.286960,4.384325,0.053333,19.491944


## 9. Example Composer Input

This simulates the dashboard inputs. Change these values to test different prospective songs.

In [12]:
composer_input = {
    "danceability": 0.70,
    "energy": 0.75,
    "loudness": -6.5,
    "speechiness": 0.06,
    "acousticness": 0.18,
    "instrumentalness": 0.00,
    "liveness": 0.12,
    "valence": 0.62,
    "tempo": 120.0,
    "duration_min": 3.4,
    "explicit_int": 0,
    "key": 1,
    "mode": 1,
    "time_signature": 4,
    "track_genre": "pop",
}

composer_df = pd.DataFrame([composer_input])
composer_df

,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_min,explicit_int,key,mode,time_signature,track_genre
0,0.7,0.75,-6.5,0.06,0.18,0.0,0.12,0.62,120.0,3.4,0,1,1,4,pop


## 10. Output 1: Similar Existing Songs

In [13]:
similar_songs = recommend_songs(
    input_profile=composer_input,
    scaler=recommender_scaler,
    recommender=recommender,
    song_catalog=song_catalog,
    top_n=10,
    genre=composer_input["track_genre"],
)

similar_songs[[
    "track_name", "artists", "track_genre", "popularity", "similarity_score",
    "danceability", "energy", "valence", "tempo"
]]

,track_name,artists,track_genre,popularity,similarity_score,danceability,energy,valence,tempo
67823,"Aafat (From ""Liger"") - Remix By DJ Aqeel",Tanishk Bagchi;Zahrah S Khan;Dj Aqeel,pop,45,0.941931,0.738,0.892,0.683,127.974
67916,"Gali Gali (From ""Kgf Chapter 1"")",Neha Kakkar;Tanishk Bagchi,pop,66,0.934238,0.789,0.950,0.746,126.013
67678,"Kusu Kusu (From ""Satyameva Jayate 2"")",Tanishk Bagchi;Zahrah S Khan;Dev Negi,pop,71,0.927338,0.742,0.911,0.780,101.000
67980,Wait a Minute!,WILLOW,pop,69,0.927320,0.764,0.705,0.672,101.003
67655,Take My Breath - Single Version,The Weeknd,pop,0,0.923960,0.747,0.739,0.523,120.996
67674,"Najaa (From ""Sooryavanshi"")",Tanishk Bagchi;Pav Dharia;Nikhita Gandhi,pop,70,0.912288,0.786,0.727,0.697,93.970
67695,What Makes You Beautiful,One Direction,pop,84,0.911827,0.726,0.787,0.888,124.990
67934,Drag Me Down,One Direction,pop,80,0.910095,0.730,0.703,0.595,138.113
67733,Loved By You,Justin Bieber;Burna Boy,pop,0,0.900387,0.790,0.724,0.642,101.040
67827,Mere Dil Gaaye Ja (Zooby Zooby),Tanishk Bagchi;Zahrah S Khan;Yash Narvekar;Bap...,pop,18,0.890571,0.673,0.902,0.618,129.024


## 11. Output 2: Base Popularity Prediction

This is the fame-free estimate. In the dashboard, the band is usually more reliable than over-interpreting exact score differences.

In [14]:
base_popularity_prediction, popularity_label = predict_base_popularity(composer_input, final_popularity_model)

print(f"Base popularity prediction: {base_popularity_prediction:.2f} / 100")
print(f"Popularity band: {popularity_label}")

Base popularity prediction: 57.93 / 100
Popularity band: Moderate


## 12. Output 3: Recommended Artists To Pitch To

The artist-assisted score combines base popularity, artist residual lift, and style similarity. Treat it as a decision-support estimate, not a guarantee of virality.

In [15]:
artist_recommendations = recommend_artists(
    input_profile=composer_input,
    artist_profiles=artist_profiles,
    base_popularity_prediction=base_popularity_prediction,
    scaler=recommender_scaler,
    top_n=10,
    genre=composer_input["track_genre"],
)

artist_recommendations

,artist,track_count,main_genre,avg_popularity,artist_lift_shrunk,style_similarity,artist_assisted_popularity,ranking_score
404,One Direction,5,pop,83.000000,7.380876,0.957325,64.998256,83.702244
1499,Zahrah S Khan,6,pop,58.000000,2.404353,0.947189,60.209739,78.968794
786,Tanishk Bagchi,37,pop,60.086957,4.776781,0.924210,62.347113,78.661760
953,Harrdy Sandhu,7,pop,65.000000,4.058549,0.917904,61.657721,78.564936
1559,Jaani,5,pop,64.666667,2.265819,0.868747,59.900786,75.213031
1140,Daft Punk,8,pop,50.818182,3.452374,0.805977,60.714896,70.660777
5072,John Newman,8,pop,13.086957,-4.133951,0.910042,54.170292,70.320619
5453,MistaJam,6,pop,1.000000,-6.058314,0.935863,52.262612,69.864359
5319,Justin Bieber,136,pop,22.182353,-5.125733,0.885025,53.395961,69.583193
720,Taylor Swift,15,pop,56.105263,5.171396,0.760346,61.864411,69.082094


## 13. Save Dashboard Artifacts

These files are what the Streamlit dashboard should load later.

In [16]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(recommender_scaler, MODEL_DIR / "spotify_recommender_scaler.joblib")
joblib.dump(recommender, MODEL_DIR / "spotify_nearest_neighbors.joblib")
joblib.dump(final_popularity_model, MODEL_DIR / "spotify_base_popularity_model.joblib")
song_catalog.to_csv(MODEL_DIR / "spotify_song_catalog.csv", index=False)
artist_profiles.to_csv(MODEL_DIR / "spotify_artist_profiles.csv", index=False)
model_comparison.to_csv(MODEL_DIR / "spotify_popularity_model_comparison.csv", index=False)

metadata = build_model_metadata(
    df=df,
    song_catalog=song_catalog,
    artist_profiles=artist_profiles,
    model_comparison=model_comparison,
    selected_model_name=selected_model_name,
)
(MODEL_DIR / "spotify_model_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")

print(f"Saved model artifacts to: {MODEL_DIR}")

Saved model artifacts to: ../models


## 14. Modeling Interpretation

The selected base model is useful for a prototype, but the score must be positioned correctly. A moderate `R?` is expected because popularity depends on many hidden social and commercial variables. For the dashboard, use the exact score as an estimate and the Low/Moderate/High band as the more stable user-facing interpretation. The artist-fit layer should remain separate from the base score so the composer can distinguish song potential from artist amplification.